# AutoQSAR Benchmark Summary

This notebook reads the output from `run_autoqsar_ga_benchmarks.py` and summarizes model performance across datasets.

By default it loads the most recent run under `./benchmark_results/`, but you can point it at any benchmark output directory.

In [12]:
# @title 0. Choose a benchmark run { display-mode: "form" }
benchmark_run_dir = "AUTO"  # @param {type:"string"}
show_top_n_models = 10  # @param {type:"slider", min:3, max:20, step:1}
rmse_metric_column_preference = "AUTO"  # @param {type:"string"}
prediction_sample_cap = 5000  # @param {type:"integer"}


In [13]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


def display_note(text):
    display(Markdown(text))


def resolve_benchmark_dir(path_text="AUTO"):
    path_text = str(path_text).strip()
    if path_text and path_text.upper() != "AUTO":
        candidate = Path(path_text).expanduser().resolve()
        if not candidate.exists():
            raise FileNotFoundError(f"Benchmark directory not found: {candidate}")
        return candidate

    root = Path("./benchmark_results").resolve()
    if not root.exists():
        raise FileNotFoundError("No ./benchmark_results directory exists yet.")
    candidates = sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError("No benchmark run directories were found under ./benchmark_results.")
    return candidates[0]


def read_optional_csv(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    return None


def read_optional_json(path):
    path = Path(path)
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return None


def pick_column(columns, candidates):
    cols = list(columns)
    for candidate in candidates:
        if candidate in cols:
            return candidate
    lowered = {str(col).strip().lower(): col for col in cols}
    for candidate in candidates:
        match = lowered.get(str(candidate).strip().lower())
        if match is not None:
            return match
    return None


def metric_column(metric_df, preferred="AUTO", metric_hint="rmse"):
    if metric_df is None or metric_df.empty:
        return None
    if str(preferred).strip() and str(preferred).strip().upper() != "AUTO":
        if preferred not in metric_df.columns:
            raise ValueError(f"Requested metric column not found: {preferred}")
        return preferred
    candidates = []
    hint = str(metric_hint).strip().lower()
    if hint == "rmse":
        candidates = [
            "test_rmse", "Test RMSE", "cv_rmse", "CV RMSE", "mean_test_rmse",
            "rmse", "RMSE"
        ]
    elif hint == "r2":
        candidates = ["test_r2", "Test R2", "cv_r2", "CV R2", "r2", "R2"]
    return pick_column(metric_df.columns, candidates)


def standardize_metric_frame(metric_df):
    if metric_df is None:
        return None
    frame = metric_df.copy()
    rename_map = {}
    dataset_col = pick_column(frame.columns, ["dataset", "dataset_id", "dataset_name"])
    model_col = pick_column(frame.columns, ["model", "Model"])
    workflow_col = pick_column(frame.columns, ["workflow", "Workflow"])
    if dataset_col is not None:
        rename_map[dataset_col] = "dataset"
    if model_col is not None:
        rename_map[model_col] = "model"
    if workflow_col is not None:
        rename_map[workflow_col] = "workflow"
    frame = frame.rename(columns=rename_map)
    return frame


In [14]:
RUN_DIR = resolve_benchmark_dir(benchmark_run_dir)
SUMMARY_METRICS = standardize_metric_frame(read_optional_csv(RUN_DIR / "summary_metrics.csv"))
TEST_RMSE_PIVOT = read_optional_csv(RUN_DIR / "test_rmse_pivot.csv")
PREDICTIONS = read_optional_csv(RUN_DIR / "predictions.csv")
GA_HISTORY = read_optional_csv(RUN_DIR / "ga_history.csv")
RUN_CONFIG = read_optional_json(RUN_DIR / "run_config.json")

dataset_metric_tables = []
for metrics_path in sorted(RUN_DIR.glob("*/metrics.csv")):
    dataset_name = metrics_path.parent.name
    frame = pd.read_csv(metrics_path)
    frame = standardize_metric_frame(frame)
    if frame is not None and not frame.empty and "dataset" not in frame.columns:
        frame["dataset"] = dataset_name
    dataset_metric_tables.append(frame)

if SUMMARY_METRICS is None and dataset_metric_tables:
    SUMMARY_METRICS = pd.concat(dataset_metric_tables, ignore_index=True)

if SUMMARY_METRICS is None or SUMMARY_METRICS.empty:
    raise ValueError(f"No metrics tables were found under {RUN_DIR}")

if "dataset" not in SUMMARY_METRICS.columns:
    raise ValueError("Could not identify a dataset column in the metrics table.")
if "model" not in SUMMARY_METRICS.columns:
    raise ValueError("Could not identify a model column in the metrics table.")

RMSE_COL = metric_column(SUMMARY_METRICS, rmse_metric_column_preference, metric_hint="rmse")
R2_COL = metric_column(SUMMARY_METRICS, "AUTO", metric_hint="r2")

print(f"Benchmark run directory: {RUN_DIR}")
print(f"Datasets: {SUMMARY_METRICS['dataset'].nunique()}")
print(f"Models: {SUMMARY_METRICS['model'].nunique()}")
print(f"Rows in summary table: {len(SUMMARY_METRICS):,}")
print(f"RMSE column used: {RMSE_COL}")
if R2_COL is not None:
    print(f"R2 column used: {R2_COL}")
if RUN_CONFIG is not None:
    display_note("Loaded `run_config.json` for this benchmark run.")


Benchmark run directory: C:\Users\Scott.Coffin\OneDrive - California OEHHA\R_new\AutoQSAR\benchmark_results\autoqsar_benchmark_20260409_111856
Datasets: 2
Models: 5
Rows in summary table: 10
RMSE column used: test_rmse
R2 column used: test_r2


Loaded `run_config.json` for this benchmark run.

In [15]:
config_summary = pd.DataFrame([RUN_CONFIG]) if isinstance(RUN_CONFIG, dict) else pd.DataFrame()
if not config_summary.empty:
    display(config_summary.T.rename(columns={0: "value"}))

best_by_dataset = SUMMARY_METRICS.sort_values(RMSE_COL, ascending=True).groupby("dataset", as_index=False).first()
best_by_dataset = best_by_dataset[[col for col in ["dataset", "workflow", "model", RMSE_COL, R2_COL] if col in best_by_dataset.columns]]
display_note("### Best model per dataset")
display(best_by_dataset.sort_values(RMSE_COL, ascending=True).reset_index(drop=True))

model_summary = (
    SUMMARY_METRICS.groupby([col for col in ["workflow", "model"] if col in SUMMARY_METRICS.columns], dropna=False)
    .agg(
        datasets=("dataset", "nunique"),
        mean_rmse=(RMSE_COL, "mean"),
        median_rmse=(RMSE_COL, "median"),
        std_rmse=(RMSE_COL, "std"),
        best_rmse=(RMSE_COL, "min"),
        worst_rmse=(RMSE_COL, "max"),
        mean_r2=(R2_COL, "mean") if R2_COL is not None else (RMSE_COL, "size"),
    )
    .reset_index()
)
if R2_COL is None and "mean_r2" in model_summary.columns:
    model_summary = model_summary.drop(columns=["mean_r2"])
display_note("### Cross-dataset model summary")
display(model_summary.sort_values("mean_rmse", ascending=True).head(int(show_top_n_models)).reset_index(drop=True))


,value
dataset,None
include_local_csv,None
output_dir,C:\Users\Scott.Coffin\OneDrive - California OE...
dry_run,False
minimum_rows,20
row_limit,0
fingerprint_bits,2048
log10_target,True
split_strategy,target_quartiles
test_fraction,0.2


### Best model per dataset

,dataset,workflow,model,test_rmse,test_r2
0,chemml_organic_density,conventional,ElasticNetCV,0.006536,0.956249
1,chemml_cep_homo,conventional,CatBoost,0.126980,0.967520


### Cross-dataset model summary

,workflow,model,datasets,mean_rmse,median_rmse,std_rmse,best_rmse,worst_rmse,mean_r2
0,conventional,CatBoost,2,0.067349,0.067349,0.084331,0.007718,0.126980,0.953257
1,conventional,ElasticNetCV,2,0.068051,0.068051,0.086995,0.006536,0.129566,0.961217
2,conventional,XGBoost,2,0.073814,0.073814,0.092255,0.008580,0.139048,0.942827
3,conventional,Random forest,2,0.074003,0.074003,0.090289,0.010160,0.137847,0.928004
4,conventional,SVR,2,0.111425,0.111425,0.109605,0.033923,0.188928,0.374748


In [16]:
# TDC leaderboard comparison (if internet access is available)
TDC_LEADERBOARD_URLS = {
    'caco2_wang': 'https://tdcommons.ai/benchmark/admet_group/01caco2/',
    'lipophilicity_astrazeneca': 'https://tdcommons.ai/benchmark/admet_group/05lipo/',
    'solubility_aqsoldb': 'https://tdcommons.ai/benchmark/admet_group/06aqsol/',
    'ppbr_az': 'https://tdcommons.ai/benchmark/admet_group/08ppbr/',
    'vdss_lombardo': 'https://tdcommons.ai/benchmark/admet_group/09vdss/',
    'half_life_obach': 'https://tdcommons.ai/benchmark/admet_group/16halflife/',
    'clearance_hepatocyte_az': 'https://tdcommons.ai/benchmark/admet_group/17clhepa/',
    'clearance_microsome_az': 'https://tdcommons.ai/benchmark/admet_group/18clmicro/',
    'ld50_zhu': 'https://tdcommons.ai/benchmark/admet_group/19ld50/',
}

def fetch_tdc_leaderboard_best(url):
    import pandas as pd
    tables = pd.read_html(url)
    summary_table = None
    leaderboard_table = None
    for table in tables:
        cols = [str(col).strip() for col in table.columns]
        if {'Dataset', 'Metric', 'Dataset Split'}.issubset(set(cols)):
            summary_table = table.copy()
        if {'Rank', 'Model'}.issubset(set(cols)):
            leaderboard_table = table.copy()
            break
    if leaderboard_table is None:
        return None
    leaderboard_table.columns = [str(col).strip() for col in leaderboard_table.columns]
    leaderboard_table = leaderboard_table.dropna(how='all')
    if leaderboard_table.empty:
        return None
    top = leaderboard_table.iloc[0]
    metric_name = None
    dataset_split = None
    if summary_table is not None and not summary_table.empty:
        summary_table.columns = [str(col).strip() for col in summary_table.columns]
        metric_name = str(summary_table.iloc[0].get('Metric', '')).strip() or None
        dataset_split = str(summary_table.iloc[0].get('Dataset Split', '')).strip() or None
    if metric_name is None:
        metric_name = next((col for col in leaderboard_table.columns if col not in {'Rank','Model','Contact','Link','#Params'}), None)
    metric_value = None if metric_name is None else str(top.get(metric_name, '')).strip()
    return {
        'rank': str(top.get('Rank', '')).strip(),
        'metric_name': metric_name,
        'metric_value': metric_value,
        'dataset_split': dataset_split,
        'model': str(top.get('Model', '')).strip(),
        'url': url,
    }

def normalize_metric_name(text):
    if text is None:
        return None
    t = str(text).strip().lower()
    if 'rmse' in t:
        return 'rmse'
    if 'mae' in t:
        return 'mae'
    if 'spearman' in t:
        return 'spearman'
    if 'pearson' in t:
        return 'pearson'
    if t in {'r2','r^2'}:
        return 'r2'
    return t

def metric_direction(metric_key):
    if metric_key in {'rmse','mae'}:
        return 'lower'
    if metric_key in {'r2','spearman','pearson'}:
        return 'higher'
    return 'lower'

tdc_rows = []
for dataset in sorted(SUMMARY_METRICS['dataset'].unique()):
    if not str(dataset).startswith('tdc_'):
        continue
    dataset_key = str(dataset).replace('tdc_', '', 1)
    url = TDC_LEADERBOARD_URLS.get(dataset_key)
    if not url:
        continue
    try:
        best = fetch_tdc_leaderboard_best(url)
    except Exception as exc:
        best = None
    if not best or not best.get('metric_value'):
        continue
    metric_key = normalize_metric_name(best.get('metric_name'))
    direction = metric_direction(metric_key)
    value = pd.to_numeric(pd.Series([best['metric_value']]), errors='coerce').iloc[0]
    if pd.isna(value):
        continue
    subset = SUMMARY_METRICS[SUMMARY_METRICS['dataset'] == dataset].copy()
    # choose metric column aligned to leaderboard metric
    metric_col = None
    if metric_key == 'rmse':
        metric_col = metric_column(subset, 'AUTO', metric_hint='rmse')
    elif metric_key == 'mae':
        metric_col = pick_column(subset.columns, ['test_mae','Test MAE','cv_mae','CV MAE'])
    elif metric_key == 'r2':
        metric_col = metric_column(subset, 'AUTO', metric_hint='r2')
    elif metric_key in {'spearman','pearson'}:
        metric_col = pick_column(subset.columns, ['primary_metric_value','primary_metric','cv_primary'])
    if metric_col is None or metric_col not in subset.columns:
        continue
    # pick best model according to leaderboard direction
    if direction == 'lower':
        best_row = subset.sort_values(metric_col, ascending=True).iloc[0]
        rel = (float(best_row[metric_col]) - float(value)) / float(value)
    else:
        best_row = subset.sort_values(metric_col, ascending=False).iloc[0]
        rel = (float(value) - float(best_row[metric_col])) / float(value)
    tdc_rows.append({
        'dataset': dataset,
        'leaderboard_metric': metric_key,
        'leaderboard_value': float(value),
        'leaderboard_model': best.get('model'),
        'leaderboard_rank': best.get('rank'),
        'best_model': best_row['model'],
        'best_value': float(best_row[metric_col]),
        'relative_gap': float(rel),
        'leaderboard_url': best.get('url'),
    })

if tdc_rows:
    tdc_df = pd.DataFrame(tdc_rows)
    display_note('### PyTDC leaderboard comparison (relative gap to SOTA)')
    display(tdc_df.sort_values('relative_gap', ascending=True).reset_index(drop=True))
    fig = px.bar(tdc_df, x='dataset', y='relative_gap', color='leaderboard_metric', title='Relative gap to PyTDC leaderboard (lower is better)')
    fig.update_layout(yaxis_title='Relative gap vs leaderboard')
    fig.show()
else:
    display_note('No PyTDC leaderboard comparison available (missing internet access or columns).')


No PyTDC leaderboard comparison available (missing internet access or columns).

In [ ]:
# PyTDC SOTA reference lines on Spearman/MAE plots
spearman_col = pick_column(SUMMARY_METRICS.columns, ['test_spearman','Test Spearman','cv_spearman','CV Spearman','spearman','Spearman'])
mae_col = pick_column(SUMMARY_METRICS.columns, ['test_mae','Test MAE','cv_mae','CV MAE','mae','MAE'])

if spearman_col is None and mae_col is None:
    display_note('Spearman/MAE columns were not found in the summary table.')
else:
    for metric_col, metric_name in [(spearman_col, 'spearman'), (mae_col, 'mae')]:
        if metric_col is None:
            continue
        tdc_datasets = [d for d in SUMMARY_METRICS['dataset'].unique() if str(d).startswith('tdc_')]
        for dataset in sorted(tdc_datasets):
            dataset_key = str(dataset).replace('tdc_', '', 1)
            url = TDC_LEADERBOARD_URLS.get(dataset_key)
            if not url:
                continue
            try:
                best = fetch_tdc_leaderboard_best(url)
            except Exception:
                best = None
            if not best or not best.get('metric_value'):
                continue
            lb_metric = normalize_metric_name(best.get('metric_name'))
            if lb_metric != metric_name:
                continue
            lb_value = pd.to_numeric(pd.Series([best['metric_value']]), errors='coerce').iloc[0]
            if pd.isna(lb_value):
                continue
            subset = SUMMARY_METRICS[SUMMARY_METRICS['dataset'] == dataset].copy()
            subset = subset.sort_values(metric_col, ascending=(metric_name == 'mae'))
            fig = px.bar(
                subset,
                x='model',
                y=metric_col,
                color='workflow' if 'workflow' in subset.columns else None,
                title=f'{dataset} {metric_name} by model (leaderboard SOTA dashed)',
            )
            fig.add_hline(y=float(lb_value), line_dash='dash', line_color='black', annotation_text='TDC SOTA', annotation_position='top left')
            fig.update_layout(yaxis_title=metric_col)
            fig.show()


In [17]:
heatmap_source = TEST_RMSE_PIVOT.copy() if TEST_RMSE_PIVOT is not None else None
if heatmap_source is None or heatmap_source.empty:
    heatmap_source = SUMMARY_METRICS.pivot_table(index="dataset", columns="model", values=RMSE_COL, aggfunc="mean").reset_index()

if "dataset" not in heatmap_source.columns:
    first_col = heatmap_source.columns[0]
    heatmap_source = heatmap_source.rename(columns={first_col: "dataset"})

heatmap_df = heatmap_source.set_index("dataset")
fig = px.imshow(
    heatmap_df,
    aspect="auto",
    color_continuous_scale="Viridis_r",
    labels={"x": "Model", "y": "Dataset", "color": RMSE_COL},
    text_auto='.3f',
    title="Held-out RMSE by dataset and model",
)
fig.update_layout(height=max(450, 60 + 40 * len(heatmap_df.index)))
fig.show()

box_df = SUMMARY_METRICS.copy()
fig = px.box(
    box_df,
    x="model",
    y=RMSE_COL,
    color="workflow" if "workflow" in box_df.columns else None,
    points="all",
    title="RMSE distribution across datasets",
)
fig.update_layout(xaxis_title="Model", yaxis_title=RMSE_COL)
fig.show()

bar_df = model_summary.sort_values("mean_rmse", ascending=True).head(int(show_top_n_models)).copy()
fig = px.bar(
    bar_df,
    x="model",
    y="mean_rmse",
    color="workflow" if "workflow" in bar_df.columns else None,
    error_y="std_rmse" if "std_rmse" in bar_df.columns else None,
    title="Mean held-out RMSE by model",
)
fig.update_layout(xaxis_title="Model", yaxis_title="Mean RMSE")
fig.show()


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [18]:
if PREDICTIONS is not None and not PREDICTIONS.empty:
    pred_df = PREDICTIONS.copy()
    dataset_col = pick_column(pred_df.columns, ["dataset", "dataset_id", "dataset_name"])
    model_col = pick_column(pred_df.columns, ["model", "Model"])
    observed_col = pick_column(pred_df.columns, ["observed", "y_true", "actual"])
    predicted_col = pick_column(pred_df.columns, ["predicted", "y_pred", "prediction"])
    split_col = pick_column(pred_df.columns, ["split", "Split"])

    if all(col is not None for col in [dataset_col, model_col, observed_col, predicted_col]):
        pred_df = pred_df.rename(columns={
            dataset_col: "dataset",
            model_col: "model",
            observed_col: "observed",
            predicted_col: "predicted",
        })
        if split_col is not None:
            pred_df = pred_df.rename(columns={split_col: "split"})
            pred_df = pred_df.loc[pred_df["split"].astype(str).str.lower() == "test"].copy()
        if len(pred_df) > int(prediction_sample_cap):
            pred_df = pred_df.sample(int(prediction_sample_cap), random_state=42)

        fig = px.scatter(
            pred_df,
            x="observed",
            y="predicted",
            color="model",
            facet_col="dataset",
            facet_col_wrap=3,
            opacity=0.65,
            title="Observed vs predicted on held-out rows",
        )
        low = float(np.nanmin([pred_df["observed"].min(), pred_df["predicted"].min()]))
        high = float(np.nanmax([pred_df["observed"].max(), pred_df["predicted"].max()]))
        fig.add_trace(go.Scatter(x=[low, high], y=[low, high], mode="lines", name="Ideal", line=dict(color="black", dash="dash")))
        fig.show()
    else:
        display_note("Predictions table exists, but the expected observed/predicted columns were not found.")
else:
    display_note("No aggregate predictions table was found for this run.")

if GA_HISTORY is not None and not GA_HISTORY.empty:
    ga_df = GA_HISTORY.copy()
    dataset_col = pick_column(ga_df.columns, ["dataset", "dataset_id", "dataset_name"])
    model_col = pick_column(ga_df.columns, ["model", "Model"])
    generation_col = pick_column(ga_df.columns, ["generation", "Generation"])
    best_col = pick_column(ga_df.columns, ["best_fitness", "best_rmse", "best_score", "Best fitness"])
    if all(col is not None for col in [dataset_col, model_col, generation_col, best_col]):
        ga_df = ga_df.rename(columns={dataset_col: "dataset", model_col: "model", generation_col: "generation", best_col: "best_value"})
        fig = px.line(ga_df, x="generation", y="best_value", color="model", facet_col="dataset", facet_col_wrap=3, title="GA convergence history")
        fig.update_layout(yaxis_title=best_col)
        fig.show()
    else:
        display_note("GA history exists, but the expected columns were not found for plotting.")


No aggregate predictions table was found for this run.